# Quantum Neural Networks Tutorial

This notebook demonstrates how to use Quantum Neural Networks (QNN) for classification tasks.

**Key Concepts:**
1. **Feature Encoding**: Classical data → Quantum states
2. **Variational Learning**: Parameterized quantum circuits learn optimal transformations
3. **Measurement**: Quantum states → Classical predictions
4. **Gradient Descent**: Parameter shift rule for quantum gradients

**Author**: Quantum Data Structures Research  
**Date**: November 2025

In [ ]:
# Imports
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from sim.q_neural_network import (
    QuantumNeuralNetwork,
    create_synthetic_dataset,
    train_test_split
)

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
np.random.seed(42)

## 1. Create Synthetic Dataset

We'll create a binary classification dataset with 4 features.

In [ ]:
# Create dataset
X, y = create_synthetic_dataset(
    n_samples=200,
    n_features=4,
    n_classes=2,
    noise=0.2,
    random_state=42
)

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Dataset Shape:")
print(f"  Training:   {X_train.shape} samples")
print(f"  Test:       {X_test.shape} samples")
print(f"  Features:   {X_train.shape[1]}")
print(f"  Classes:    {len(np.unique(y))}")
print(f"\nClass Distribution (train): {np.bincount(y_train)}")
print(f"Class Distribution (test):  {np.bincount(y_test)}")

In [ ]:
# Visualize first two features
plt.figure(figsize=(10, 6))
plt.scatter(X_train[y_train==0, 0], X_train[y_train==0, 1], 
            c='red', label='Class 0', alpha=0.6, edgecolors='k')
plt.scatter(X_train[y_train==1, 0], X_train[y_train==1, 1], 
            c='blue', label='Class 1', alpha=0.6, edgecolors='k')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Training Data (First 2 Features)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. Initialize Quantum Neural Network

We'll create a QNN with:
- **4 qubits** (matching 4 features)
- **2 variational layers**
- **Angle encoding** for feature map
- **Hardware-efficient ansatz** for variational circuit

In [ ]:
# Initialize QNN
qnn = QuantumNeuralNetwork(
    n_qubits=4,
    n_layers=2,
    feature_map="angle",
    ansatz="hardware_efficient",
    learning_rate=0.1,
    batch_size=20,
    shots=1024
)

print(f"QNN Architecture:")
print(f"  Qubits:          {qnn.n_qubits}")
print(f"  Layers:          {qnn.n_layers}")
print(f"  Parameters:      {qnn.n_params}")
print(f"  Feature Map:     {qnn.feature_map_type}")
print(f"  Ansatz:          {qnn.ansatz_type}")
print(f"  Learning Rate:   {qnn.learning_rate}")
print(f"  Batch Size:      {qnn.batch_size}")
print(f"  Shots:           {qnn.shots}")

## 3. Visualize Circuit Architecture

Let's see what the quantum circuit looks like for a sample input.

In [ ]:
# Create circuit for sample input
sample_x = X_train[0]
qc = qnn.forward(sample_x)

print(f"Circuit for sample input:")
print(f"  Depth:     {qc.depth()}")
print(f"  Gates:     {qc.size()}")
print(f"  Qubits:    {qc.num_qubits}")
print(f"\nCircuit diagram:")
print(qc.draw(output='text', fold=-1))

## 4. Train the QNN

We'll train using the **Adam optimizer** for 50 epochs.

In [ ]:
# Train QNN
print("Training Quantum Neural Network...\n")

history = qnn.train(
    X_train, y_train,
    X_val=X_test, y_val=y_test,
    epochs=50,
    loss_type="cross_entropy",
    optimizer="adam",
    verbose=True
)

print("\nTraining complete!")

## 5. Visualize Training Progress

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history['train_accuracy'], label='Train Accuracy', linewidth=2)
axes[1].plot(history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Evaluate Model Performance

In [ ]:
# Make predictions
train_predictions = qnn.predict_batch(X_train)
test_predictions = qnn.predict_batch(X_test)

# Calculate accuracy
train_accuracy = np.mean(train_predictions == y_train)
test_accuracy = np.mean(test_predictions == y_test)

print("Final Performance:")
print(f"  Train Accuracy: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
print(f"  Test Accuracy:  {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

# Confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

print("\nConfusion Matrix (Test):")
cm = confusion_matrix(y_test, test_predictions)
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, test_predictions, target_names=['Class 0', 'Class 1']))

## 7. Visualize Learned Quantum Features

The QNN learns to map inputs to quantum states. We can extract these quantum features.

In [ ]:
# Extract learned features for test set
quantum_features = qnn.get_learned_features(X_test)

print(f"Quantum Feature Representation:")
print(f"  Original features: {X_test.shape[1]}")
print(f"  Quantum features:  {quantum_features.shape[1]} (2^{qnn.n_qubits})")
print(f"\nSample quantum features (first 10 dims):")
print(quantum_features[0, :10])

In [ ]:
# Visualize quantum features using t-SNE
from sklearn.manifold import TSNE

# Reduce to 2D for visualization
tsne = TSNE(n_components=2, random_state=42)
features_2d = tsne.fit_transform(quantum_features)

plt.figure(figsize=(10, 6))
plt.scatter(features_2d[y_test==0, 0], features_2d[y_test==0, 1],
            c='red', label='Class 0', alpha=0.6, edgecolors='k')
plt.scatter(features_2d[y_test==1, 0], features_2d[y_test==1, 1],
            c='blue', label='Class 1', alpha=0.6, edgecolors='k')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.title('Learned Quantum Features (t-SNE Projection)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. Compare with Classical Neural Network

In [ ]:
# Train classical neural network for comparison
from sklearn.neural_network import MLPClassifier

classical_nn = MLPClassifier(
    hidden_layer_sizes=(16, 16),
    max_iter=50,
    random_state=42
)

classical_nn.fit(X_train, y_train)
classical_train_acc = classical_nn.score(X_train, y_train)
classical_test_acc = classical_nn.score(X_test, y_test)

print("\nComparison: Quantum vs Classical")
print("=" * 50)
print(f"Model              | Train Acc | Test Acc")
print("-" * 50)
print(f"Quantum NN         | {train_accuracy:.4f}    | {test_accuracy:.4f}")
print(f"Classical NN       | {classical_train_acc:.4f}    | {classical_test_acc:.4f}")
print("=" * 50)

# Bar plot
models = ['Quantum NN', 'Classical NN']
train_accs = [train_accuracy, classical_train_acc]
test_accs = [test_accuracy, classical_test_acc]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width/2, train_accs, width, label='Train', alpha=0.8)
ax.bar(x + width/2, test_accs, width, label='Test', alpha=0.8)

ax.set_ylabel('Accuracy')
ax.set_title('Quantum vs Classical Neural Network Performance')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 9. Experiment with Different Architectures

In [ ]:
# Test different ansatzes
ansatzes = ["hardware_efficient", "strongly_entangling", "real_amplitudes"]
results = {}

print("Testing different ansatzes...\n")

for ansatz in ansatzes:
    print(f"Training with {ansatz}...")
    
    qnn_test = QuantumNeuralNetwork(
        n_qubits=4,
        n_layers=2,
        ansatz=ansatz,
        learning_rate=0.1,
        batch_size=20,
        shots=1024
    )
    
    history = qnn_test.train(
        X_train, y_train,
        X_val=X_test, y_val=y_test,
        epochs=30,
        optimizer="adam",
        verbose=False
    )
    
    test_preds = qnn_test.predict_batch(X_test)
    test_acc = np.mean(test_preds == y_test)
    
    results[ansatz] = {
        'accuracy': test_acc,
        'n_params': qnn_test.n_params,
        'final_loss': history['val_loss'][-1]
    }
    
    print(f"  Accuracy: {test_acc:.4f}, Parameters: {qnn_test.n_params}\n")

# Compare results
print("\nAnsatz Comparison:")
print("=" * 70)
print(f"{'Ansatz':<25} | {'Params':<8} | {'Accuracy':<10} | {'Loss':<10}")
print("-" * 70)
for ansatz, result in results.items():
    print(f"{ansatz:<25} | {result['n_params']:<8} | {result['accuracy']:<10.4f} | {result['final_loss']:<10.4f}")
print("=" * 70)

## 10. Save Trained Model

In [ ]:
# Save the best model
model_path = "../results/qnn_model.npz"
qnn.save_model(model_path)

print(f"Model saved to: {model_path}")
print(f"\nModel contains:")
print(f"  - Trained parameters ({qnn.n_params} values)")
print(f"  - Architecture configuration")
print(f"  - Training history")

## Summary

In this tutorial, we:

1. ✅ Created a synthetic binary classification dataset
2. ✅ Initialized a Quantum Neural Network with 4 qubits and 2 layers
3. ✅ Visualized the quantum circuit architecture
4. ✅ Trained the QNN using gradient descent with parameter shift rule
5. ✅ Evaluated performance on test data
6. ✅ Extracted learned quantum feature representations
7. ✅ Compared with classical neural networks
8. ✅ Experimented with different ansatz architectures
9. ✅ Saved the trained model

### Key Insights:

- **Feature Learning**: The QNN learns to transform classical data into quantum states that separate classes
- **Decision Boundaries**: Implicit in the quantum measurement process
- **Quantum Advantage**: Exponential Hilbert space (2^n dimensions) from n qubits
- **Trade-offs**: Quantum circuits require more shots for stable gradients

### Next Steps:

1. Try on real datasets (MNIST, CIFAR-10)
2. Experiment with deeper circuits
3. Add noise models for realistic simulation
4. Deploy on real quantum hardware
5. Compare with quantum kernel methods